In [ ]:
from fastapi import FastAPI, HTTPException

from backend.data_loader import matches, deliveries

from backend.analysis import (
    get_total_matches,
    get_total_deliveries,
    get_seasons,
    get_teams,
    get_matches_by_season,
    get_team_wins,
    get_toss_analysis,
    get_top_batsmen,
    get_top_six_hitters,
    get_top_four_hitters,
    get_top_bowlers,

    get_team_statistics,
    get_team_season_performance,
    get_player_batting_stats,
    get_player_bowling_stats,
    get_season_statistics,
    get_venue_statistics

)


# =========================================================
# CREATE FASTAPI APPLICATION
# =========================================================

app = FastAPI(
    title="IPL Analytics API",
    description="FastAPI backend for IPL Analytics Dashboard",
    version="1.0.0"
)


# =========================================================
# HOME
# =========================================================

@app.get("/")
def home():

    return {
        "message": "Welcome to IPL Analytics API",
        "status": "running"
    }


# =========================================================
# SUMMARY
# =========================================================

@app.get("/summary")
def summary():

    return {
        "total_matches": get_total_matches(matches),
        "total_deliveries": get_total_deliveries(
            deliveries
        ),
        "total_teams": len(
            get_teams(matches)
        ),
        "total_seasons": len(
            get_seasons(matches)
        )
    }


# =========================================================
# SEASONS
# =========================================================

@app.get("/seasons")
def seasons():

    return {
        "seasons": get_seasons(matches)
    }


# =========================================================
# TEAMS
# =========================================================

@app.get("/teams")
def teams():

    return {
        "teams": get_teams(matches)
    }


# =========================================================
# MATCHES BY SEASON
# =========================================================

@app.get("/matches-by-season")
def matches_by_season():

    data = get_matches_by_season(matches)

    return data.to_dict(
        orient="records"
    )


# =========================================================
# TEAM WINS
# =========================================================

@app.get("/team-wins")
def team_wins():

    data = get_team_wins(matches)

    return data.to_dict(
        orient="records"
    )


# =========================================================
# TOSS ANALYSIS
# =========================================================

@app.get("/toss-analysis")
def toss_analysis():

    data = get_toss_analysis(matches)

    return data.to_dict(
        orient="records"
    )


# =========================================================
# TOP BATSMEN
# =========================================================

@app.get("/top-batsmen")
def top_batsmen():

    data = get_top_batsmen(
        deliveries,
        top_n=10
    )

    return data.to_dict(
        orient="records"
    )


# =========================================================
# TOP SIX HITTERS
# =========================================================

@app.get("/top-six-hitters")
def top_six_hitters():

    data = get_top_six_hitters(
        deliveries,
        top_n=10
    )

    return data.to_dict(
        orient="records"
    )


# =========================================================
# TOP FOUR HITTERS
# =========================================================

@app.get("/top-four-hitters")
def top_four_hitters():

    data = get_top_four_hitters(
        deliveries,
        top_n=10
    )

    return data.to_dict(
        orient="records"
    )


# =========================================================
# TOP BOWLERS
# =========================================================

@app.get("/top-bowlers")
def top_bowlers():

    data = get_top_bowlers(
        deliveries,
        top_n=10
    )

    return data.to_dict(
        orient="records"
    )

# =========================================================
# TEAM STATISTICS
# =========================================================

@app.get("/team/{team_name}")
def team_statistics(team_name: str):

    teams = get_teams(matches)

    matched_team = next(
        (team for team in teams
         if team.lower() == team_name.lower()),
        None
    )

    if matched_team is None:
        raise HTTPException(
            status_code=404,
            detail=f"Team '{team_name}' not found"
        )

    return get_team_statistics(
        matches,
        matched_team
    )

# =========================================================
# TEAM SEASON STATISTICS
# =========================================================

@app.get("/team/{team_name}/season")
def team_season_statistics(
    team_name: str,
    season: str
):
    teams = get_teams(matches)

    matched_team = next(
        (team for team in teams
         if team.lower() == team_name.lower()),
        None
    )

    if matched_team is None:
        raise HTTPException(
            status_code=404,
            detail=f"Team '{team_name}' not found"
        )

    if season not in get_seasons(matches):
        raise HTTPException(
            status_code=404,
            detail=f"Season '{season}' not found"
        )

    return get_team_season_performance(
        matches,
        matched_team,
        season
    )

# =========================================================
# PLAYER BATTING STATISTICS
# =========================================================


@app.get("/players")
def players():
    players = (
        deliveries["batter"]
        .dropna()
        .unique()
        .tolist()
    )

    return {
        "players": sorted(players)
    }



# =========================================================
# PLAYER BATTING STATISTICS
# =========================================================

@app.get("/player/{player_name}/batting")
def player_batting(player_name: str):

    players = deliveries["batter"].dropna().unique()

    matched_player = next(
        (player for player in players
         if player.lower() == player_name.lower()),
        None
    )

    if matched_player is None:
        raise HTTPException(
            status_code=404,
            detail=f"Player '{player_name}' not found"
        )

    return get_player_batting_stats(
        deliveries,
        matched_player
    )


# =========================================================
# PLAYER BOWLING STATISTICS
# =========================================================

@app.get("/player/{player_name}/bowling")
def player_bowling(player_name: str):

    players = deliveries["bowler"].dropna().unique()

    matched_player = next(
        (player for player in players
         if player.lower() == player_name.lower()),
        None
    )

    if matched_player is None:
        raise HTTPException(
            status_code=404,
            detail=f"Player '{player_name}' not found"
        )

    return get_player_bowling_stats(
        deliveries,
        matched_player
    )


# =========================================================
# SEASON STATISTICS
# =========================================================

@app.get("/season/{season}")
def season_statistics(season: str):

    if season not in get_seasons(matches):
        raise HTTPException(
            status_code=404,
            detail=f"Season '{season}' not found"
        )

    return get_season_statistics(
        matches,
        season
    )


# =========================================================
# VENUE STATISTICS
# =========================================================

@app.get("/venue-statistics")
def venue_statistics():

    data = get_venue_statistics(
        matches
    )

    return data.to_dict(
        orient="records"
    )
